# Verifier Output Analysis

This notebook inspects the output of the v1 Paninian verifier batch run.

**Input:** `results/verifier_batch_meaningful.csv` (rows where `decision_type` is not `no_decision`)

**Source script:** `src/verifier/run_verifier_batch.py` (first 50 sentences from Hindi-HDTB train)

This is analysis only — **no verifier logic is modified here.**

## 1. Load the CSV

In [1]:
import csv
from collections import Counter
from pathlib import Path

CSV_PATH = Path("../results/verifier_batch_meaningful.csv")


def load_verifier_csv(filepath):
    """Load verifier batch CSV as a list of row dictionaries."""
    with open(filepath, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


rows = load_verifier_csv(CSV_PATH)

print(f"Loaded {len(rows)} meaningful rows from {CSV_PATH}")
if rows:
    print(f"Columns: {', '.join(rows[0].keys())}")

Loaded 36 meaningful rows from ..\results\verifier_batch_meaningful.csv
Columns: sent_id, sentence_text, token_form, deprel, case_marker, rule_id, decision_type, confidence, karaka_candidates, reason


## 2. Count by Rule

In [2]:
rule_counts = Counter(row["rule_id"] for row in rows)

print(f"{'Rule':<8} {'Count':>8}")
print("-" * 18)
for rule_id, count in sorted(rule_counts.items()):
    print(f"{rule_id:<8} {count:>8}")

Rule        Count
------------------
R1              3
R2             15
R3              1
R4             11
R5              6


## 3. Count by Decision Type

In [3]:
decision_counts = Counter(row["decision_type"] for row in rows)

print(f"{'Decision type':<16} {'Count':>8}")
print("-" * 26)
for decision_type, count in sorted(decision_counts.items()):
    print(f"{decision_type:<16} {count:>8}")

Decision type       Count
--------------------------
ambiguous              17
confirmed              19


## 4. Example Rows by Rule

Up to **five examples** per rule (R1–R5).

In [4]:
DISPLAY_COLS = [
    "sent_id",
    "token_form",
    "deprel",
    "case_marker",
    "rule_id",
    "decision_type",
    "karaka_candidates",
]


def show_examples(example_rows, title, max_examples=5):
    """Print up to max_examples rows in a readable format."""
    print(title)
    print("=" * 70)

    if not example_rows:
        print("(no rows)")
        print()
        return

    for i, row in enumerate(example_rows[:max_examples], start=1):
        print(f"Example {i}")
        print(f"  Sentence: {row['sentence_text']}")
        for col in DISPLAY_COLS:
            print(f"  {col}: {row[col]}")
        print()


RULE_ORDER = ["R1", "R2", "R3", "R4", "R5"]

for rule_id in RULE_ORDER:
    rule_rows = [row for row in rows if row["rule_id"] == rule_id]
    show_examples(rule_rows, title=f"Rule {rule_id} — {len(rule_rows)} total row(s)")

Rule R1 — 3 total row(s)
Example 1
  Sentence: इसे नवाब शाहजेहन ने बनवाया था ।
  sent_id: train-s2
  token_form: शाहजेहन
  deprel: nsubj
  case_marker: ने
  rule_id: R1
  decision_type: confirmed
  karaka_candidates: Kartā

Example 2
  Sentence: इसे चार्ल्स कोरिया ने डिजाइन किया है ।
  sent_id: train-s11
  token_form: कोरिया
  deprel: nsubj
  case_marker: ने
  rule_id: R1
  decision_type: confirmed
  karaka_candidates: Kartā

Example 3
  Sentence: उनके आगे के राजाओं ने शानदार इमारतें और भवन बनवाकर इसकी शान में चार चाँद लगा दिए ।
  sent_id: train-s50
  token_form: राजाओं
  deprel: nsubj
  case_marker: ने
  rule_id: R1
  decision_type: confirmed
  karaka_candidates: Kartā

Rule R2 — 15 total row(s)
Example 1
  Sentence: जिसमें चार मेहराबें हैं और मुख्य प्रार्थना हॉल में जाने के लिए 9 प्रवेश द्वार हैं ।
  sent_id: train-s4
  token_form: हॉल
  deprel: obl
  case_marker: में
  rule_id: R2
  decision_type: confirmed
  karaka_candidates: Adhikaraṇa

Example 2
  Sentence: विशाल क्षेत्र में फैल

## 5. Five Example Ambiguous Rows

In [5]:
ambiguous_rows = [row for row in rows if row["decision_type"] == "ambiguous"]
show_examples(ambiguous_rows, title=f"Ambiguous decisions — {len(ambiguous_rows)} total row(s)")

Ambiguous decisions — 17 total row(s)
Example 1
  Sentence: यहाँ लगने वाला तीन दिन का इज्तिमा पूरे देश के लोगों को आमंत्रित करता है ।
  sent_id: train-s6
  token_form: लोगों
  deprel: obj
  case_marker: को
  rule_id: R5
  decision_type: ambiguous
  karaka_candidates: Karma|Sampradāna

Example 2
  Sentence: मुख्य रूप से यह प्रदर्शन कला और दृश्य कला का केंद्र है ।
  sent_id: train-s10
  token_form: रूप
  deprel: obl
  case_marker: से
  rule_id: R4
  decision_type: ambiguous
  karaka_candidates: Karaṇa|Apādāna

Example 3
  Sentence: यह एक प्रागैतिहासिक स्थल पर है और विश्व में अपनी तरह का एक ही संग्रहालय है जो प्रागैतिहासिक चित्रकला से सज्जित गुफाओं के समीप है ।
  sent_id: train-s16
  token_form: चित्रकला
  deprel: obl
  case_marker: से
  rule_id: R4
  decision_type: ambiguous
  karaka_candidates: Karaṇa|Apādāna

Example 4
  Sentence: और इस तरह से यह वस्तुओं और परंपराओं से जीवंत रूप से जुड़ा हुआ है ।
  sent_id: train-s17
  token_form: तरह
  deprel: obl
  case_marker: से
  rule_id: R4
  dec

## 6. Five Example Confirmed Rows

In [6]:
confirmed_rows = [row for row in rows if row["decision_type"] == "confirmed"]
show_examples(confirmed_rows, title=f"Confirmed decisions — {len(confirmed_rows)} total row(s)")

Confirmed decisions — 19 total row(s)
Example 1
  Sentence: इसे नवाब शाहजेहन ने बनवाया था ।
  sent_id: train-s2
  token_form: शाहजेहन
  deprel: nsubj
  case_marker: ने
  rule_id: R1
  decision_type: confirmed
  karaka_candidates: Kartā

Example 2
  Sentence: जिसमें चार मेहराबें हैं और मुख्य प्रार्थना हॉल में जाने के लिए 9 प्रवेश द्वार हैं ।
  sent_id: train-s4
  token_form: हॉल
  deprel: obl
  case_marker: में
  rule_id: R2
  decision_type: confirmed
  karaka_candidates: Adhikaraṇa

Example 3
  Sentence: इसे चार्ल्स कोरिया ने डिजाइन किया है ।
  sent_id: train-s11
  token_form: कोरिया
  deprel: nsubj
  case_marker: ने
  rule_id: R1
  decision_type: confirmed
  karaka_candidates: Kartā

Example 4
  Sentence: विशाल क्षेत्र में फैले इस भवन के आस - पास का प्राकृतिक सौंदर्य इसे और भी भव्य बनाता है ।
  sent_id: train-s12
  token_form: क्षेत्र
  deprel: obl
  case_marker: में
  rule_id: R2
  decision_type: confirmed
  karaka_candidates: Adhikaraṇa

Example 5
  Sentence: यह एक अनूठा संग्रहालय ह

## Summary

This notebook reviewed meaningful verifier outputs from a 50-sentence batch.

Use these examples to:
- Check whether R1–R5 fire on plausible tokens
- Inspect ambiguous cases (R4, R5) for future disambiguation rules
- Document observations in `docs/research_notes.md`

To regenerate the CSV, run:

```bash
python src/verifier/run_verifier_batch.py
```